# Goal models

## 1. Poisson model
Each team has an attack and a defence strength. Goals follow a Poisson law with log(expected goals) = mean + attack + opponent defence + home advantage.
I fit it by weighted maximum likelihood on the 8 previous years, with a 2-year half-life, and refit it at the start of every month of the validation period.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np

from src.database import query
from src.evaluation import compare
from src.models.elo_logistic import EloLogistic
from src.models.poisson import PoissonModel, rolling_predict
from src.splits import load_matches

matches = load_matches()
train = matches[matches["split"] == "train"]
valid = matches[matches["split"] == "valid"]
history = query("""
    SELECT date, home_team, away_team, home_score, away_score, neutral
    FROM clean_results
    ORDER BY date, match_id
""")

In [2]:
predictions = {
    "elo_logistic": EloLogistic().fit(train).predict_proba(valid),
    "poisson": rolling_predict(lambda: PoissonModel(), history, valid),
}
compare(predictions, valid["result"])

,log_loss,brier,rps
elo_logistic,0.8847,0.5197,0.1747
poisson,0.8743,0.5157,0.1731


## 2. Experiment 1: regularisation
I change only `alpha`, with the half-life fixed at 730 days and the window at 8 years.

In [3]:
import pandas as pd

from src.evaluation import evaluate

rows = []
for alpha in [0.01, 0.001, 0.0001, 0.00001, 0.000001]:
    probs = rolling_predict(lambda: PoissonModel(alpha=alpha), history, valid)
    rows.append({"alpha": alpha, **evaluate(probs, valid["result"])})
pd.DataFrame(rows).round({"log_loss": 4, "brier": 4, "rps": 4})

,alpha,log_loss,brier,rps
0,0.010000,0.9657,0.5723,0.1992
1,0.001000,0.8969,0.5282,0.1789
2,0.000100,0.8759,0.5164,0.1735
3,0.000010,0.8743,0.5157,0.1731
4,0.000001,0.8745,0.5158,0.1732


Lower regularisation improves every metric until a plateau at alpha = 0.00001 (log loss 0.8743). With a strong penalty, team strengths are pulled toward the average and the probabilities are too flat.
I keep alpha = 0.00001: it is the best score, and a small penalty still protects teams with very few matches.
The Poisson model now beats the Elo logistic baseline (0.8743 vs 0.8847).

## 3. Experiment 2: half-life
I change only the half-life, with alpha fixed at 0.00001 and the window at 8 years.

In [4]:
rows = []
for half_life in [180, 365, 730, 1095, 1460]:
    probs = rolling_predict(lambda: PoissonModel(half_life_days=half_life, alpha=0.00001), history, valid)
    rows.append({"half_life_days": half_life, **evaluate(probs, valid["result"])})
pd.DataFrame(rows).round({"log_loss": 4, "brier": 4, "rps": 4})

,half_life_days,log_loss,brier,rps
0,180,0.9073,0.5315,0.1800
1,365,0.8791,0.5178,0.1741
2,730,0.8743,0.5157,0.1731
3,1095,0.8754,0.5166,0.1735
4,1460,0.8767,0.5175,0.1738


The half-life shows a U shape: 180 days is too short (few effective matches, noisy strengths), 3 or 4 years react too slowly to changes in team level. 730 days is the best (log loss 0.8743).
The best half-life is the one used to tune alpha, so both choices are consistent.
Final Poisson settings: alpha = 0.00001, half-life 730 days, window 8 years.

## 4. Dixon-Coles
Dixon and Coles (1997) correct the four low scores (0-0, 1-0, 0-1, 1-1) with one parameter rho, because independent Poisson underestimates 0-0 and 1-1 in English league data.
I estimate rho in a second step, on the Poisson expected goals, at each monthly refit.

In [5]:
from src.models.dixon_coles import DixonColesModel

predictions = {
    "elo_logistic": EloLogistic().fit(train).predict_proba(valid),
    "poisson": rolling_predict(lambda: PoissonModel(), history, valid),
    "dixon_coles": rolling_predict(lambda: DixonColesModel(), history, valid),
}
compare(predictions, valid["result"])

,log_loss,brier,rps
elo_logistic,0.8847,0.5197,0.1747
poisson,0.8743,0.5157,0.1731
dixon_coles,0.8747,0.5159,0.1731


In [6]:
DixonColesModel().fit(history, "2024-01-01").rho

np.float64(-0.03519971780312951)

In [7]:
from src.models.poisson import score_matrix

periods = valid["date"].dt.to_period("M")
matrices = np.zeros((len(valid), 11, 11))
for period in periods.unique():
    in_period = (periods == period).to_numpy()
    model = PoissonModel().fit(history, period.start_time)
    home_goals, away_goals = model.expected_goals(valid[in_period])
    matrices[in_period] = score_matrix(home_goals, away_goals)

rows = []
for home, away in [(0, 0), (1, 0), (0, 1), (1, 1), (2, 2)]:
    observed = ((valid["home_score"] == home) & (valid["away_score"] == away)).mean()
    rows.append({"score": f"{home}-{away}", "predicted": np.nanmean(matrices[:, home, away]), "observed": observed})
pd.DataFrame(rows).round(4)

,score,predicted,observed
0,0-0,0.0955,0.0758
1,1-0,0.1134,0.1118
2,0-1,0.0957,0.0901
3,1-1,0.0983,0.1076
4,2-2,0.0295,0.0387


Dixon-Coles does not improve on Poisson (log loss 0.8747 vs 0.8743), and the estimated rho is close to 0 (-0.035).
On international matches, Poisson overestimates 0-0 (9.6% vs 7.6%) but underestimates 1-1 and 2-2. Dixon-Coles moves 0-0 and 1-1 in the same direction, so the two effects cancel out.
The total draw probability is already right (22.8% predicted vs 22.7% observed), because the errors inside draws compensate. I keep the simpler Poisson model as the goal model. A bivariate Poisson with diagonal inflation is a possible improvement.